### Carte combinée — Précipitations et oxygène dissous

- Representation conjointe des précipitations et des concentrations d'oxygène dissous dans la Senne.

- Données du 24 juillet 2025

- L'objectif est de créer une animation qui montre l'évolution des deux phénomènes au fil du temps, puis de lui associer une sonification.

Charger les données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

# Charger les mesures de pluie et d'oxygène
mesures = pd.read_csv(
    "../data/24_juillet_2025_donnees_preparees.csv"
)

# Harmoniser le nom du pluviomètre Écluse Anderlecht
mesures = mesures.rename(columns={
    "pluv.écluse;anderlecht": "pluv.écluse-anderlecht"
})

# Charger les coordonnées des capteurs d'oxygène
stations_oxygene = pd.read_csv(
    "../data/coordonnees_oxygene_senne.csv"
)

# Charger le tracé de la Senne
senne = gpd.read_file(
    "../data/tracé_senne.geojson"
)

print("Nombre de mesures :", len(mesures))
print("Colonnes disponibles :")
print(mesures.columns.tolist())

print("Nombre de capteurs d'oxygène :", len(stations_oxygene))
print("Tracé de la Senne chargé :", len(senne), "élément(s)")

Préparer les données de pluie et d'oxygène (séparer les deux types de données)

In [ ]:
# Identifier les colonnes de pluie
colonnes_pluie = [
    colonne for colonne in mesures.columns
    if colonne.startswith("pluv.") or colonne == "Pluv_Buda"
]

# Identifier les colonnes d'oxygène
colonnes_oxygene = ["Q01", "Q02", "Q03", "Q04"]

# Préparer les valeurs numériques
pluie = mesures[colonnes_pluie].apply(
    pd.to_numeric, errors="coerce"
)

oxygene = mesures[colonnes_oxygene].apply(
    pd.to_numeric, errors="coerce"
)

# Vérifier les données
print("Nombre de stations de pluie :", len(colonnes_pluie))
print("Nombre de capteurs d'oxygène :", len(colonnes_oxygene))
print("Dimensions des données de pluie :", pluie.shape)
print("Dimensions des données d'oxygène :", oxygene.shape)

Charger les coordonnées de pluie

In [ ]:
# Recharger les fichiers pour repartir des données originales
mesures = pd.read_csv(
    "../data/24_juillet_2025_donnees_preparees.csv"
)

stations_pluie = pd.read_csv(
    "../data/coordonnees_pluviometres_flowbru.csv"
)

# Faire correspondre les noms des colonnes
mesures.columns = mesures.columns.str.strip()
stations_pluie["colonne_pluie_csv"] = (
    stations_pluie["colonne_pluie_csv"].str.strip()
)

# Retrouver automatiquement la colonne Écluse Anderlecht
colonne_ecluse = next(
    c for c in mesures.columns
    if "anderlecht" in c.lower()
)

# Lui donner exactement le nom utilisé dans le fichier des coordonnées
nom_correct = stations_pluie.loc[
    stations_pluie["id_flowbru"] == "P07",
    "colonne_pluie_csv"
].iloc[0]

mesures = mesures.rename(columns={
    colonne_ecluse: nom_correct
})

# Préparer les 16 colonnes de pluie
colonnes_pluie = stations_pluie["colonne_pluie_csv"].tolist()

# Vérifier avant de sélectionner les colonnes
manquantes = set(colonnes_pluie) - set(mesures.columns)
print("Colonnes manquantes :", manquantes)

if not manquantes:
    pluie = mesures[colonnes_pluie].apply(
        pd.to_numeric, errors="coerce"
    )
    print("Dimensions de pluie :", pluie.shape)

Verification nome pluviometre ecluse de Anderlecht

In [ ]:
# Comparer les noms exacts de la station Écluse Anderlecht
for colonne in mesures.columns:
    if "cluse" in colonne.lower():
        print("Mesures :", repr(colonne))

for colonne in stations_pluie["colonne_pluie_csv"]:
    if "cluse" in colonne.lower():
        print("Coordonnées :", repr(colonne))

Afficher la Senne, la pluie et l'oxygène sur une même carte

In [ ]:
# Transformer les coordonnées des pluviomètres en points géographiques
points_pluie = gpd.GeoDataFrame(
    stations_pluie,
    geometry=gpd.points_from_xy(
        stations_pluie["x_lambert72"],
        stations_pluie["y_lambert72"]
    ),
    crs="EPSG:31370"
)

# Vérifier les colonnes des stations d'oxygène et le système de coordonnées
print("Colonnes oxygène :", stations_oxygene.columns.tolist())
print("Système de coordonnées de la Senne :", senne.crs)

Créer la carte avec les 20 stations

In [ ]:
# Transformer les coordonnées des capteurs d'oxygène en points
points_oxygene = gpd.GeoDataFrame(
    stations_oxygene,
    geometry=gpd.points_from_xy(
        stations_oxygene["x_lambert72"],
        stations_oxygene["y_lambert72"]
    ),
    crs="EPSG:31370"
)

# Créer la figure
fig, ax = plt.subplots(figsize=(10, 10))

# Dessiner la Senne
senne.plot(
    ax=ax,
    color="steelblue",
    linewidth=2,
    label="Senne"
)

# Dessiner les 16 pluviomètres
points_pluie.plot(
    ax=ax,
    color="orange",
    markersize=55,
    edgecolor="black",
    label="Pluviomètres"
)

# Dessiner les 4 capteurs d'oxygène
points_oxygene.plot(
    ax=ax,
    color="red",
    markersize=90,
    marker="s",
    edgecolor="black",
    label="Capteurs d'oxygène"
)

# Ajouter les noms des capteurs d'oxygène
for _, station in points_oxygene.iterrows():
    ax.annotate(
        station["capteur"],
        (station.geometry.x, station.geometry.y),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

ax.set_title("Bruxelles — Pluviomètres et capteurs d'oxygène")
ax.set_xlabel("Coordonnée X (Lambert 72)")
ax.set_ylabel("Coordonnée Y (Lambert 72)")
ax.set_aspect("equal")
ax.legend()

plt.show()

#### Première essaye visualisation sans son
Faire varier les points selon les mesures. 
Pour cette première version :
- Pluie (orange) : plus il pleut, plus le cercle est grand.
- Oxygène (bleu) : la couleur représente la concentration en oxygène dissous.

Visualisation des mesures — Premier instant

In [ ]:
# Choisir le premier instant de la journée
instant = 0

# Récupérer les mesures de pluie dans l'ordre des stations
valeurs_pluie = pluie.iloc[instant].to_numpy()

# Récupérer les mesures d'oxygène dans l'ordre des capteurs
valeurs_oxygene = oxygene.loc[
    instant,
    stations_oxygene["capteur"].tolist()
].to_numpy(dtype=float)

# Créer la carte
fig, ax = plt.subplots(figsize=(10, 10))

# Tracé de la Senne
senne.plot(ax=ax, color="steelblue", linewidth=2)

# Pluviomètres : taille selon la quantité de pluie
ax.scatter(
    points_pluie.geometry.x,
    points_pluie.geometry.y,
    s=40 + valeurs_pluie * 250,
    color="orange",
    edgecolor="black",
    alpha=0.7,
    label="Pluie"
)

# Oxygène : couleur selon la concentration
points = ax.scatter(
    points_oxygene.geometry.x,
    points_oxygene.geometry.y,
    c=valeurs_oxygene,
    cmap="viridis",
    vmin=0,
    vmax=12,
    s=130,
    marker="s",
    edgecolor="black",
    label="Oxygène"
)

# Échelle des couleurs pour l'oxygène
plt.colorbar(
    points,
    ax=ax,
    label="Oxygène dissous (mg/L)"
)

ax.set_title("Pluie et oxygène — Premier instant")
ax.set_aspect("equal")
ax.legend()

plt.show()

Animation de la carte combinée — Pluie et oxygène

On a ici la carte animée avec :
- Les cercles orange qui grossissent lorsqu'il pleut.
- Les carrés dont la couleur change selon la concentration d'oxygène.
- L'heure qui avance toutes les cinq minutes.
- Le tracé de la Senne qui reste fixe.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Préparer les mesures dans l'ordre des stations
valeurs_pluie = pluie[
    stations_pluie["colonne_pluie_csv"].tolist()
].to_numpy(dtype=float)

valeurs_oxygene = oxygene[
    stations_oxygene["capteur"].tolist()
].to_numpy(dtype=float)

# Préparer la figure
fig, ax = plt.subplots(figsize=(10, 10))

# Dessiner la Senne
senne.plot(
    ax=ax,
    color="steelblue",
    linewidth=2
)

# Créer les points de pluie
cercles_pluie = ax.scatter(
    points_pluie.geometry.x,
    points_pluie.geometry.y,
    s=np.full(16, 40),
    color="orange",
    edgecolor="black",
    alpha=0.7,
    label="Pluie"
)

# Créer les points d'oxygène
carres_oxygene = ax.scatter(
    points_oxygene.geometry.x,
    points_oxygene.geometry.y,
    c=valeurs_oxygene[0],
    cmap="viridis",
    vmin=0,
    vmax=12,
    s=130,
    marker="s",
    edgecolor="black",
    label="Oxygène"
)

# Ajouter l'échelle des couleurs
fig.colorbar(
    carres_oxygene,
    ax=ax,
    label="Oxygène dissous (mg/L)"
)

ax.set_aspect("equal")
ax.legend()

# Fonction qui actualise la carte
def actualiser(i):

    # Faire varier la taille des cercles de pluie
    pluie_instant = np.nan_to_num(
        valeurs_pluie[i],
        nan=0
    )

    cercles_pluie.set_sizes(
        40 + pluie_instant * 250
    )

    # Faire varier la couleur des carrés d'oxygène
    carres_oxygene.set_array(
        valeurs_oxygene[i]
    )

    # Afficher l'heure correspondante
    heure = mesures.iloc[i]["Date/heure locale (UTC+2)"]

    ax.set_title(
        f"Bruxelles — Pluie et oxygène\n{heure}"
    )

    return cercles_pluie, carres_oxygene

# Créer l'animation
animation = FuncAnimation(
    fig,
    actualiser,
    frames=len(mesures),
    interval=100,
    blit=False
)

plt.close(fig)

# Afficher l'animation dans le notebook
HTML(animation.to_jshtml())

#### Deuxième essaye : animation avec des barres pour l'oxygène

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Préparer les mesures dans l'ordre des stations
valeurs_pluie = pluie[
    stations_pluie["colonne_pluie_csv"].tolist()
].to_numpy(dtype=float)

valeurs_oxygene = oxygene[
    stations_oxygene["capteur"].tolist()
].to_numpy(dtype=float)

# Créer la figure
fig, ax = plt.subplots(figsize=(10, 10))

# Dessiner la Senne
senne.plot(
    ax=ax,
    color="steelblue",
    linewidth=2
)

# Pluviomètres : cercles orange
cercles_pluie = ax.scatter(
    points_pluie.geometry.x,
    points_pluie.geometry.y,
    s=np.full(16, 40),
    color="orange",
    edgecolor="black",
    alpha=0.7,
    label="Pluie"
)

# Préparer les barres d'oxygène
barres_oxygene = []

# Échelle visuelle : 1 mg/L = 150 mètres sur la carte
echelle_oxygene = 150

for _, station in points_oxygene.iterrows():

    x = station.geometry.x
    y = station.geometry.y

    # Une barre verticale à l'emplacement de chaque sonde
    barre, = ax.plot(
        [x, x],
        [y, y],
        color="darkblue",
        linewidth=7,
        solid_capstyle="butt"
    )

    barres_oxygene.append(barre)

    # Nom de la sonde
    ax.annotate(
        station["capteur"],
        (x, y),
        xytext=(8, -12),
        textcoords="offset points",
        fontsize=9
    )

# Garder une carte fixe pendant l'animation
ax.set_xlim(
    min(points_pluie.geometry.x.min(), points_oxygene.geometry.x.min()) - 1000,
    max(points_pluie.geometry.x.max(), points_oxygene.geometry.x.max()) + 1000
)

ax.set_ylim(
    min(points_pluie.geometry.y.min(), points_oxygene.geometry.y.min()) - 1000,
    max(points_pluie.geometry.y.max(), points_oxygene.geometry.y.max()) + 2500
)

ax.set_aspect("equal")
ax.legend()

# Fonction d'animation
def actualiser(i):

    # Pluie : modifier la taille des cercles
    pluie_instant = np.nan_to_num(
        valeurs_pluie[i],
        nan=0
    )

    cercles_pluie.set_sizes(
        40 + pluie_instant * 250
    )

    # Oxygène : modifier la hauteur des barres
    for j, barre in enumerate(barres_oxygene):

        x = points_oxygene.geometry.x.iloc[j]
        y = points_oxygene.geometry.y.iloc[j]

        valeur = valeurs_oxygene[i, j]

        if np.isnan(valeur):
            hauteur = 0
        else:
            hauteur = max(0, valeur) * echelle_oxygene

        barre.set_data(
            [x, x],
            [y, y + hauteur]
        )

    # Afficher l'heure
    heure = mesures.iloc[i]["Date/heure locale (UTC+2)"]

    ax.set_title(
        f"Bruxelles — Pluie et oxygène\n{heure}"
    )

    return [cercles_pluie] + barres_oxygene

# Créer l'animation
animation = FuncAnimation(
    fig,
    actualiser,
    frames=len(mesures),
    interval=100,
    blit=False
)

plt.close(fig)

HTML(animation.to_jshtml())

Exporter l'animation combinée

In [ ]:
from pathlib import Path
from matplotlib.animation import FFMpegWriter
import imageio_ffmpeg
import matplotlib as mpl

# Utiliser FFmpeg comme dans le notebook 06
mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

# Dossier de sortie
dossier_sortie = Path("../outputs")
dossier_sortie.mkdir(parents=True, exist_ok=True)

# Vidéo combinée sans son
video_combinee = dossier_sortie / "pluie_oxygene_sans_son.mp4"

# Durée : 30 secondes pour 288 images
duree_video = 60
fps = len(mesures) / duree_video

writer = FFMpegWriter(
    fps=fps,
    codec="libx264",
    bitrate=2500
)

animation.save(
    str(video_combinee),
    writer=writer,
    dpi=120
)

print("Vidéo créée :", video_combinee)

In [ ]:
import subprocess
from IPython.display import Audio, display

# Réutiliser le son du notebook 06
audio = Path("../audio/senne_oxygene.wav")

# Nouvelle vidéo avec son
sortie = Path("../outputs/pluie_oxygene_avec_son.mp4")

if not audio.exists():
    print("Fichier audio introuvable :", audio)
else:
    subprocess.run([
        imageio_ffmpeg.get_ffmpeg_exe(),
        "-y",
        "-i", str(video_combinee),
        "-i", str(audio),
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        str(sortie)
    ], check=True)

    print("Vidéo avec son créée :", sortie)
    display(Audio(filename=str(audio)))

Ajouter le son existant pour obtenir une première vidéo complète

Préparation des données pour la sonification. Calculer deux séries temporelles :
- Pluie : la moyenne des mesures des 16 pluviomètres à chaque instant.
- Oxygène : la moyenne des mesures des 4 sondes à chaque instant.

Cett etape permet de associer, avec STRAUSS, la pluie à un paramètre sonore (par exemple l'intensité) et l'oxygène à un autre (par exemple la hauteur du son).

In [ ]:
# Calculer les moyennes à chaque instant
pluie_moyenne = pluie.mean(axis=1)
oxygene_moyen = oxygene.mean(axis=1)

# Préparer un tableau pour la sonification
donnees_son = pd.DataFrame({
    "temps": mesures["Date/heure locale (UTC+2)"],
    "pluie": pluie_moyenne,
    "oxygene": oxygene_moyen
})

# Afficher les premières lignes
display(donnees_son.head())

print("Nombre d'instants :", len(donnees_son))

Verification outils STRAUSS

In [ ]:
from strauss.sonification import Sonification
from strauss.sources import Events
from strauss.generator import Synthesizer

print("Les outils de sonification STRAUSS sont disponibles !")

### Sonification oxygene et pluie:
Les paramètres sonores doivent correspondre aux informations qu'on cherche à transmettre, et il faut vérifier que les auditeurs perçoivent effectivement ces informations. Question à se poser:
1. Peut-on reconnaître quelle station est active ?
2. Peut-on entendre si la pluie ou l'oxygène augmente ou diminue ?
3. Peut-on repérer un éventuel décalage temporel entre les précipitations et les variations d'oxygène ?


##### Choix des paramètres de sonification


**Remarque méthodologique :** Ces correspondances entre données et paramètres sonores constituent des hypothèses de conception. Elles doivent être expérimentées et évaluées pour déterminer si elles permettent de distinguer les stations et de percevoir les variations des mesures.


| Élément | Proposition | Pourquoi |
|---|---|---|
| Identité de la station | Position stéréo et/ou timbre distinct | Reconnaître les différentes stations |
| Intensité de la pluie | Fréquence des impulsions | Évoquer les gouttes et percevoir l'intensification |
| Concentration d'oxygène | Hauteur d'un son continu | Entendre les hausses et les baisses |
| Évolution temporelle | Synchronisation avec l'animation | Relier ce qu'on entend à ce qu'on voit |


Expérience sonore 1 — Quatre sondes d'oxygène
- Chaque sonde (Q01, Q02, Q03, Q04) aura une position différente dans l'espace stéréo.
- La hauteur du son variera selon la concentration en oxygène.
- Les quatre sondes seront entendues pendant la même période, pour pouvoir comparer leurs évolutions.

In [ ]:
# Conserver les données de chaque sonde séparément
donnees_oxygene_son = oxygene[
    ["Q01", "Q02", "Q03", "Q04"]
].copy()

# Préparer un temps commun pour les 288 mesures
# Les 24 heures seront représentées en 30 secondes
duree_son = 30

donnees_oxygene_son["temps_son"] = np.linspace(
    0,
    duree_son,
    len(donnees_oxygene_son)
)

# Vérifier les données
display(donnees_oxygene_son.head())

print("Nombre de mesures :", len(donnees_oxygene_son))
print("Durée sonore prévue :", duree_son, "secondes")

Préparer les données pour STRAUSS

In [ ]:
# Les quatre sondes que nous voulons entendre
noms_sondes = ["Q01", "Q02", "Q03", "Q04"]

# Temps de chaque mesure dans notre séquence de 30 secondes
temps_son = np.linspace(
    0,
    30,
    len(oxygene)
)

# Normaliser les valeurs d'oxygène entre 0 et 1
# On utilise une échelle commune pour les quatre sondes
valeur_min = oxygene[noms_sondes].min().min()
valeur_max = oxygene[noms_sondes].max().max()

oxygene_normalise = (
    (oxygene[noms_sondes] - valeur_min)
    / (valeur_max - valeur_min)
)

# Vérifier le résultat
display(oxygene_normalise.head())

print("Minimum d'oxygène :", valeur_min)
print("Maximum d'oxygène :", valeur_max)

Vérifier le générateur sonore

In [ ]:
from strauss.sources import Objects
from strauss.generator import Synthesizer
from strauss.sonification import Sonification

import inspect

print("Paramètres de Objects :")
print(inspect.signature(Objects))

print("\nParamètres de Synthesizer :")
print(inspect.signature(Synthesizer))

print("\nParamètres de Sonification :")
print(inspect.signature(Sonification))

Vérifier comment STRAUSS organise les données

In [ ]:
print(Objects.__doc__)

Ils seront utilisé:
- Une voix sonore par sonde : Q01, Q02, Q03 et Q04.
- Une hauteur de son qui évolue avec l'oxygène.
- Une position stéréo différente pour distinguer les sondes.

Comprendre le format des données de STRAUSS

In [ ]:
import inspect

print(inspect.getsource(Objects.__init__))

##### Créer les sources sonores.

On va avoir:
- pitch : la hauteur du son, selon la concentration en oxygène.
- azimuth : la position du son dans l'espace stéréo, pour différencier les quatre sondes.

In [ ]:
# Créer les sources sonores pour les 4 sondes d'oxygène
sources_oxygene = Objects(
    mapped_quantities=["pitch", "azimuth"]
)

print("Sources sonores créées !")

Préparer la position sonore de chaque sonde

In [ ]:
# Positions stéréo des quatre sondes
positions_sondes = {
    "Q01": -0.75,
    "Q02": -0.25,
    "Q03":  0.25,
    "Q04":  0.75
}

# Préparer les données de chaque sonde
donnees_par_sonde = {}

for sonde in noms_sondes:
    donnees_par_sonde[sonde] = {
        "temps": temps_son,
        "oxygene": oxygene_normalise[sonde].to_numpy(),
        "position": positions_sondes[sonde]
    }

print("Données préparées pour :", list(donnees_par_sonde.keys()))

In [ ]:
# Afficher les méthodes disponibles pour Objects
print([
    methode for methode in dir(sources_oxygene)
    if not methode.startswith("_")
])

Examiner fromdict

In [ ]:
import inspect

print(inspect.signature(sources_oxygene.fromdict))
print(sources_oxygene.fromdict.__doc__)

In [ ]:
import inspect

print(inspect.getsource(sources_oxygene.fromdict))

Transmettre les quatre sondes à STRAUSS

In [ ]:
# Préparer les données des quatre sondes
donnees_strauss = {
    "pitch": [
        oxygene_normalise["Q01"].to_numpy(),
        oxygene_normalise["Q02"].to_numpy(),
        oxygene_normalise["Q03"].to_numpy(),
        oxygene_normalise["Q04"].to_numpy()
    ],
    "azimuth": [0.1, 0.35, 0.65, 0.9]
}

# Transmettre les données à STRAUSS
sources_oxygene.fromdict(donnees_strauss)

# Vérifier le nombre de sources
print("Nombre de sources sonores :", sources_oxygene.n_sources)

Préparer l'évolution temporelle des sons

In [ ]:
from strauss import sources

print([
    nom for nom in dir(sources)
    if "evo" in nom.lower() or "time" in nom.lower()
])

print([
    nom for nom in dir(sources_oxygene)
    if "mapping" in nom.lower()
])

Vérifier le format de l'évolution temporelle

In [ ]:
from strauss import sources

print("Paramètres qui peuvent évoluer :")
print(sources.evolvable)

print("\nFonction de correspondance :")
import inspect
print(inspect.signature(sources_oxygene.apply_mapping_functions))

Préparer les quatre voix avec les bons paramètres

In [ ]:
# Recréer les sources avec les paramètres adaptés
sources_oxygene = Objects(
    mapped_quantities=[
        "pitch_shift",
        "time_evo",
        "azimuth"
    ]
)

print("Sources préparées pour l'évolution temporelle.")


#### Paramètres STRAUSS pour la sonification de l'oxygène

| Paramètre STRAUSS | Rôle |
|---|---|
| `pitch_shift` | Faire varier la hauteur du son selon l'oxygène |
| `time_evo` | Décrire l'évolution des mesures dans le temps |
| `azimuth` | Positionner les sondes dans l'espace sonore |


Préparer les données temporelles pour STRAUSS

In [ ]:
# Temps normalisé entre 0 et 1
temps_normalise = np.linspace(0, 1, len(oxygene))

# Préparer les variations de hauteur des quatre sondes
variations_hauteur = [
    oxygene_normalise[sonde].to_numpy(dtype=float)
    for sonde in noms_sondes
]

# Préparer une évolution temporelle pour chaque sonde
evolutions_temps = [
    temps_normalise.copy()
    for sonde in noms_sondes
]

print("Nombre de sondes :", len(variations_hauteur))
print("Mesures par sonde :", len(variations_hauteur[0]))
print("Temps normalisé :", temps_normalise[0], "à", temps_normalise[-1])

Transmettre les données à STRAUSS

In [ ]:
# Transmettre les données des 4 sondes à STRAUSS
donnees_strauss = {
    "pitch_shift": variations_hauteur,
    "time_evo": evolutions_temps,
    "azimuth": [0.1, 0.35, 0.65, 0.9]
}

sources_oxygene.fromdict(donnees_strauss)

print("Nombre de sources :", sources_oxygene.n_sources)
print("Paramètres :", sources_oxygene.mapped_quantities)

Vérifier la structure avant de générer le son

In [ ]:
import inspect

print(inspect.getsource(
    sources_oxygene.apply_mapping_functions
))

Appliquer les correspondances sonores

In [ ]:
# Transformer les données en paramètres sonores
sources_oxygene.apply_mapping_functions(
    map_lims={
        "pitch_shift": (0, 1),
        "time_evo": (0, 1),
        "azimuth": (0, 1)
    },
    param_lims={
        "pitch_shift": (-12, 12),
        "time_evo": (0, 1),
        "azimuth": (0, 1)
    }
)

print("Correspondances sonores appliquées !")
print("Nombre de sondes :", sources_oxygene.n_sources)

Créer le synthétiseur

In [ ]:
# Créer un synthétiseur STRAUSS
synth_oxygene = Synthesizer()

print("Synthétiseur STRAUSS créé !")

Créer la partition STRAUSS

In [ ]:
from strauss.score import Score
import inspect

print(inspect.signature(Score))